# Notebook 02: Entrenamiento del Modelo de Predicción de Hits
**Proyecto:** Hit Predictor & Trendy Dashboard — Grupo 5  

### ¿Qué hacemos en este cuaderno?
1. **Cargar** el dataset limpio (`spotify_clean.csv`) que generó el Notebook 01.
2. **Seleccionar** las variables (features) que usará el modelo.
3. **Dividir** los datos en entrenamiento (80%) y prueba (20%).
4. **Entrenar** un modelo de clasificación (Random Forest) con scikit-learn.
5. **Evaluar** qué tan bien predice con métricas claras.
6. **Exportar** el modelo entrenado como `model.pkl` para usarlo en la app de Streamlit.

---
## Paso 1: Importar librerías
Cargamos todas las herramientas que necesitamos.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib
import os

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


---
## Paso 2: Cargar el dataset limpio
Usamos el CSV que generó el Notebook 01 de limpieza de datos.

In [2]:
df = pd.read_csv("../data/spotify_clean.csv")

print(f"Dataset cargado: {df.shape[0]} canciones, {df.shape[1]} columnas")
print(f"\nDistribución de is_hit:")
print(f"  Hits (1):    {df['is_hit'].sum()} canciones ({df['is_hit'].mean():.1%})")
print(f"  No-hits (0): {(df['is_hit'] == 0).sum()} canciones ({1 - df['is_hit'].mean():.1%})")
print(f"\nPrimeras 5 filas:")
df.head()

Dataset cargado: 4494 canciones, 31 columnas

Distribución de is_hit:
  Hits (1):    1227 canciones (27.3%)
  No-hits (0): 3267 canciones (72.7%)

Primeras 5 filas:


,energy,tempo,danceability,playlist_genre,loudness,liveness,valence,track_artist,time_signature,speechiness,...,mode,key,duration_ms,acousticness,id,playlist_subgenre,type,playlist_id,duration_min,is_hit
0,0.592,157.969,0.521,pop,-7.777,0.122,0.535,"Lady Gaga, Bruno Mars",3.0,0.0304,...,0.0,6.0,251668.0,0.3080,2plbrEY59IikOBgBGLjaoe,mainstream,audio_features,37i9dQZF1DXcBWIGoYBM5M,4.19,1
1,0.507,104.978,0.747,pop,-10.171,0.117,0.438,Billie Eilish,4.0,0.0358,...,1.0,2.0,210373.0,0.2000,6dOtVTDdiauQNBQEDOtlAB,mainstream,audio_features,37i9dQZF1DXcBWIGoYBM5M,3.51,1
2,0.808,108.548,0.554,pop,-4.169,0.159,0.372,Gracie Abrams,4.0,0.0368,...,1.0,1.0,166300.0,0.2140,7ne4VBA60CxGM75vw0EYad,mainstream,audio_features,37i9dQZF1DXcBWIGoYBM5M,2.77,1
3,0.910,112.966,0.670,pop,-4.070,0.304,0.786,Sabrina Carpenter,4.0,0.0634,...,0.0,0.0,157280.0,0.0939,1d7Ptw3qYcfpdLNL5REhtJ,mainstream,audio_features,37i9dQZF1DXcBWIGoYBM5M,2.62,1
4,0.783,149.027,0.777,pop,-4.477,0.355,0.939,"ROSÉ, Bruno Mars",4.0,0.2600,...,0.0,0.0,169917.0,0.0283,5vNRhkKd0yEAg8suGBpjeY,mainstream,audio_features,37i9dQZF1DXcBWIGoYBM5M,2.83,1


---
## Paso 3: Definir variables de entrada (X) y salida (y)

- **X (features):** Los 7 atributos numéricos que describen cómo suena una canción.
- **y (target):** La columna `is_hit` — lo que queremos predecir (1 = hit, 0 = no hit).

Estas features son las mismas que luego el usuario ajustará con los sliders en Streamlit.

In [3]:
# Definir las 7 features que usará el modelo
FEATURES = [
    'danceability',    # Qué tan bailable (0-1)
    'energy',          # Intensidad / fuerza (0-1)
    'valence',         # Alegre vs triste (0-1)
    'tempo',           # Velocidad en BPM
    'loudness',        # Volumen en dB (negativo)
    'speechiness',     # Cuánto hablan vs cantan (0-1)
    'acousticness',    # Acústico vs electrónico (0-1)
]

X = df[FEATURES]  # Variables de entrada
y = df['is_hit']  # Variable a predecir

print(f"Variables de entrada (X): {X.shape[1]} features, {X.shape[0]} filas")
print(f"Variable de salida (y): {y.shape[0]} valores")
print(f"\nFeatures seleccionadas:")
for i, feat in enumerate(FEATURES, 1):
    print(f"  {i}. {feat}: min={X[feat].min():.2f}, max={X[feat].max():.2f}, promedio={X[feat].mean():.2f}")

Variables de entrada (X): 7 features, 4494 filas
Variable de salida (y): 4494 valores

Features seleccionadas:
  1. danceability: min=0.06, max=0.98, promedio=0.62
  2. energy: min=0.00, max=1.00, promedio=0.58
  3. valence: min=0.03, max=0.99, promedio=0.48
  4. tempo: min=48.23, max=241.43, promedio=118.28
  5. loudness: min=-48.07, max=1.32, promedio=-9.49
  6. speechiness: min=0.02, max=0.93, promedio=0.10
  7. acousticness: min=0.00, max=1.00, promedio=0.35


---
## Paso 4: Dividir datos en Entrenamiento y Prueba

Separamos los datos en dos grupos:
- **80% para entrenamiento:** El modelo aprende los patrones con estos datos.
- **20% para prueba:** Evaluamos si el modelo realmente aprendió (datos que nunca vio).

Usamos `stratify=y` para que ambos grupos tengan la misma proporción de hits y no-hits.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,      # 20% para prueba
    random_state=42,     # Semilla para reproducibilidad
    stratify=y           # Mantener proporción de hits/no-hits
)

print(f"Datos de ENTRENAMIENTO: {X_train.shape[0]} canciones")
print(f"  Hits: {y_train.sum()} ({y_train.mean():.1%})")
print(f"  No-hits: {(y_train == 0).sum()} ({1 - y_train.mean():.1%})")
print(f"")
print(f"Datos de PRUEBA: {X_test.shape[0]} canciones")
print(f"  Hits: {y_test.sum()} ({y_test.mean():.1%})")
print(f"  No-hits: {(y_test == 0).sum()} ({1 - y_test.mean():.1%})")

Datos de ENTRENAMIENTO: 3595 canciones
  Hits: 982 (27.3%)
  No-hits: 2613 (72.7%)

Datos de PRUEBA: 899 canciones
  Hits: 245 (27.3%)
  No-hits: 654 (72.7%)


---
## Paso 5: Entrenar el modelo (Random Forest)

**¿Qué es un Random Forest?**  
Es un algoritmo que crea muchos "árboles de decisión" (como diagramas de flujo) y combina sus respuestas para dar una predicción final. Es robusto, fácil de usar y funciona bien con este tipo de datos.

Usamos `class_weight='balanced'` porque tenemos más no-hits (73%) que hits (27%). Esto le dice al modelo que le ponga más atención a los hits para no ignorarlos.

In [5]:
# Crear y entrenar el modelo
modelo = RandomForestClassifier(
    n_estimators=200,          # 200 árboles de decisión
    max_depth=10,              # Profundidad máxima de cada árbol
    class_weight='balanced',   # Compensar el desbalance hits vs no-hits
    random_state=42,           # Semilla para reproducibilidad
    n_jobs=-1                  # Usar todos los núcleos del procesador
)

modelo.fit(X_train, y_train)

print("Modelo entrenado exitosamente.")
print(f"Tipo: {type(modelo).__name__}")
print(f"Número de árboles: {modelo.n_estimators}")
print(f"Profundidad máxima: {modelo.max_depth}")

Modelo entrenado exitosamente.
Tipo: RandomForestClassifier
Número de árboles: 200
Profundidad máxima: 10


---
## Paso 6: Evaluar el modelo

Usamos los datos de prueba (20% que el modelo nunca vio) para medir qué tan bien predice.

**Métricas clave:**
- **Accuracy:** Porcentaje total de predicciones correctas.
- **Precision:** De las canciones que dijo "hit", ¿cuántas realmente lo eran?
- **Recall:** De las canciones que SÍ eran hit, ¿cuántas logró identificar?
- **F1-Score:** Promedio balanceado entre precision y recall.

In [6]:
# Predecir con datos de prueba
y_pred = modelo.predict(X_test)

# Accuracy general
acc = accuracy_score(y_test, y_pred)
print(f"ACCURACY GENERAL: {acc:.1%}")
print(f"(De cada 100 canciones, acierta {acc*100:.0f})")

# Reporte detallado
print(f"\n{'='*50}")
print("REPORTE DETALLADO POR CLASE")
print(f"{'='*50}")
print(classification_report(y_test, y_pred, target_names=['No-Hit (0)', 'Hit (1)']))

# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
print(f"MATRIZ DE CONFUSIÓN:")
print(f"                  Predicho No-Hit | Predicho Hit")
print(f"  Real No-Hit:    {cm[0][0]:>14} | {cm[0][1]:>12}")
print(f"  Real Hit:       {cm[1][0]:>14} | {cm[1][1]:>12}")

ACCURACY GENERAL: 64.7%
(De cada 100 canciones, acierta 65)

REPORTE DETALLADO POR CLASE
              precision    recall  f1-score   support

  No-Hit (0)       0.83      0.64      0.73       654
     Hit (1)       0.41      0.66      0.50       245

    accuracy                           0.65       899
   macro avg       0.62      0.65      0.62       899
weighted avg       0.72      0.65      0.67       899

MATRIZ DE CONFUSIÓN:
                  Predicho No-Hit | Predicho Hit
  Real No-Hit:               421 |          233
  Real Hit:                   84 |          161


---
## Paso 7: ¿Qué features son más importantes?

El Random Forest nos dice cuáles de las 7 features pesan más a la hora de decidir si una canción es hit o no. Esto es muy útil para la presentación.

In [7]:
# Importancia de cada feature
importancias = pd.Series(modelo.feature_importances_, index=FEATURES)
importancias = importancias.sort_values(ascending=False)

print("IMPORTANCIA DE CADA FEATURE (de mayor a menor):")
print(f"{'='*50}")
for feat, imp in importancias.items():
    barra = '█' * int(imp * 50)
    print(f"  {feat:<16} {imp:.3f} {barra}")

IMPORTANCIA DE CADA FEATURE (de mayor a menor):
  loudness         0.217 ██████████
  acousticness     0.165 ████████
  energy           0.149 ███████
  danceability     0.124 ██████
  speechiness      0.121 ██████
  valence          0.120 ██████
  tempo            0.104 █████


---
## Paso 8: Prueba rápida — Simular una predicción

Probamos el modelo con valores inventados, como si fuéramos el usuario usando los sliders.

In [8]:
# Simular un reggaetón tipo Bad Bunny
cancion_simulada = pd.DataFrame([{
    'danceability': 0.85,
    'energy': 0.75,
    'valence': 0.60,
    'tempo': 95,
    'loudness': -4,
    'speechiness': 0.15,
    'acousticness': 0.05,
}])

prediccion = modelo.predict(cancion_simulada)[0]
probabilidad = modelo.predict_proba(cancion_simulada)[0]

print("SIMULACIÓN: Reggaetón tipo Bad Bunny")
print(f"{'='*40}")
for col in cancion_simulada.columns:
    print(f"  {col}: {cancion_simulada[col].values[0]}")
print(f"{'='*40}")
print(f"Predicción: {'HIT' if prediccion == 1 else 'NO HIT'}")
print(f"Probabilidad de ser Hit: {probabilidad[1]:.1%}")
print(f"Probabilidad de ser No-Hit: {probabilidad[0]:.1%}")

SIMULACIÓN: Reggaetón tipo Bad Bunny
  danceability: 0.85
  energy: 0.75
  valence: 0.6
  tempo: 95
  loudness: -4
  speechiness: 0.15
  acousticness: 0.05
Predicción: HIT
Probabilidad de ser Hit: 64.2%
Probabilidad de ser No-Hit: 35.8%


In [9]:
# Simular una balada acústica triste
balada = pd.DataFrame([{
    'danceability': 0.25,
    'energy': 0.20,
    'valence': 0.10,
    'tempo': 72,
    'loudness': -14,
    'speechiness': 0.03,
    'acousticness': 0.90,
}])

prediccion2 = modelo.predict(balada)[0]
probabilidad2 = modelo.predict_proba(balada)[0]

print("SIMULACIÓN: Balada acústica triste")
print(f"{'='*40}")
for col in balada.columns:
    print(f"  {col}: {balada[col].values[0]}")
print(f"{'='*40}")
print(f"Predicción: {'HIT' if prediccion2 == 1 else 'NO HIT'}")
print(f"Probabilidad de ser Hit: {probabilidad2[1]:.1%}")
print(f"Probabilidad de ser No-Hit: {probabilidad2[0]:.1%}")

SIMULACIÓN: Balada acústica triste
  danceability: 0.25
  energy: 0.2
  valence: 0.1
  tempo: 72
  loudness: -14
  speechiness: 0.03
  acousticness: 0.9
Predicción: NO HIT
Probabilidad de ser Hit: 22.2%
Probabilidad de ser No-Hit: 77.8%


---
## Paso 9: Exportar el modelo entrenado

Guardamos el modelo como un archivo `.pkl` para que el Integrante 4 lo cargue en la app de Streamlit sin tener que re-entrenar.

In [10]:
# Crear carpeta models/ si no existe
os.makedirs("../models", exist_ok=True)

# Guardar el modelo
MODEL_PATH = "../models/model.pkl"
joblib.dump(modelo, MODEL_PATH)

# Guardar también la lista de features (para que Streamlit sepa el orden)
FEATURES_PATH = "../models/features.pkl"
joblib.dump(FEATURES, FEATURES_PATH)

# Verificar que se guardó bien
tamano = os.path.getsize(MODEL_PATH) / 1024
print(f"Modelo guardado en: {MODEL_PATH} ({tamano:.0f} KB)")
print(f"Lista de features guardada en: {FEATURES_PATH}")
print(f"\nEl modelo está listo para usarse en la app de Streamlit.")

Modelo guardado en: ../models/model.pkl (5825 KB)
Lista de features guardada en: ../models/features.pkl

El modelo está listo para usarse en la app de Streamlit.
